<a href="https://colab.research.google.com/github/compost438/Smart-Compost-Robot-Vegetable-Detection/blob/main/Vegetable_Detection_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1 — Install packages
# ============================================================
# ultralytics    : YOLOv8 training / inference framework
# onnxruntime    : run the exported .onnx model (e.g. on Raspberry Pi)
# onnxsim        : simplify the ONNX graph so CPU inference is faster
# The "-q" flag keeps the install output quiet.
!pip install ultralytics -q
!pip install --upgrade ultralytics -q
!pip install onnxruntime -q
!pip install onnxsim -q

print("✅ All packages installed!")

In [ ]:
# ============================================================
# CELL 2 — Environment info & reproducibility (run first)
# ============================================================
# Fixing the random seed makes training/splitting repeatable: the same
# code + same data will give the same train/valid/test split and the
# same shuffling every run. This matters when you compare experiments.
import os, random, sys, platform, math, json, time
from pathlib import Path
import numpy as np
import torch

SEED = 42

def set_seed(seed=SEED):
    """Fix all random number generators so results are reproducible."""
    random.seed(seed)            # Python's random module (used in our split)
    np.random.seed(seed)         # NumPy
    torch.manual_seed(seed)      # PyTorch (CPU)
    torch.cuda.manual_seed_all(seed)  # PyTorch (all GPUs)
    # Deterministic mode: slightly slower but fully reproducible.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Print the runtime so it's logged for debugging (CPU vs GPU, versions).
print("Python:", sys.version)
print("OS:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# CELL 3 — Project configuration (folders + classes)
# ============================================================
# Everything lives under one working root so there's no path confusion.
#   dataset/  -> the split train/valid/test data we generate in CELL 4
#   runs/     -> YOLO training results (weights, plots, metrics)
#   cfg/      -> our hyperparameter yaml
from datetime import datetime

ROOT      = Path('/content/compost_foodwaste')   # working root in Colab
DATASET   = ROOT / "dataset"                      # train/valid/test created here
RUNS      = ROOT / "runs"                         # training outputs saved here
CFG_DIR   = ROOT / "cfg"                          # config (yaml) files
DATA_YAML = DATASET / "data.yaml"                 # the dataset description YOLO reads

# >>> CLASS NAMES <<<
# These are the VEGETABLE / food-waste classes we are detecting.
# The order defines the class IDs in the label files:
#   0 = carrot_peels, 1 = mixed_veg, 2 = onion
# This must match the IDs in your YOLO .txt labels exactly.
CLASSES = ["carrot_peels", "mixed_veg", "onion"]

# Create the folders (no error if they already exist).
for p in [ROOT, DATASET, RUNS, CFG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATASET:", DATASET)
print("RUNS:", RUNS)
print("CFG_DIR:", CFG_DIR)
print("CLASSES:", CLASSES)

In [ ]:
# ============================================================
# CELL 4 — Split the data IN COLAB (stratified by class)
# ============================================================
# This replaces Roboflow's splitting. It reads every image + its YOLO
# label from SRC (your Google Drive folder), then splits them into
# train/valid/test. "Stratified" means each split keeps a similar mix
# of carrot_peels / mixed_veg / onion, so no class is missing from valid.
import shutil
from collections import Counter, defaultdict
from google.colab import drive

drive.mount('/content/drive')   # gives Colab access to your Drive files

# ---- SETTINGS (edit SRC to point at your data) ----
# SRC can be a flat folder of images+labels, OR an existing Roboflow
# export with train/valid/test subfolders. We search it recursively,
# so either layout works — everything is pooled and re-split.
SRC = Path('/content/drive/MyDrive/Machine Learning/compost_foodwaste/dataset')
FRAC_TRAIN, FRAC_VAL = 0.7, 0.2     # 70% train, 20% valid, remaining 10% test
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

# ---- 1) Index all label files by their base filename (stem) ----
# e.g. image_001.txt -> looked up later for image_001.jpg
all_labels = {p.stem: p for p in SRC.rglob('*.txt')}

# ---- 2) Collect all images ----
images = sorted(p for p in SRC.rglob('*') if p.suffix.lower() in IMG_EXTS)
assert images, f'No images found under {SRC} (check the SRC path)'

# ---- 3) Decide each image's "primary" class (its most common box) ----
# We use the most frequent class in an image to decide which group it
# belongs to for stratification. Images with no/empty label = background.
primary_class, negatives = {}, []
for img in images:
    lf = all_labels.get(img.stem)
    counts = Counter()
    if lf and lf.exists():
        for line in lf.read_text().splitlines():
            parts = line.split()
            if len(parts) == 5:                 # YOLO format: class cx cy w h
                counts[int(float(parts[0]))] += 1
    if counts:
        primary_class[img] = counts.most_common(1)[0][0]
    else:
        negatives.append(img)                   # no objects = background image

print(f'{len(images)} images | labeled: {len(primary_class)} | background: {len(negatives)}')

# ---- 4) Split each class separately (this is the "stratified" part) ----
by_cls = defaultdict(list)
for img, cid in primary_class.items():
    by_cls[cid].append(img)

train, val, test = [], [], []
for cid, imgs in by_cls.items():
    random.shuffle(imgs)                        # seed was fixed in CELL 2
    n = len(imgs)
    n_tr, n_va = round(FRAC_TRAIN * n), round(FRAC_VAL * n)
    train += imgs[:n_tr]
    val   += imgs[n_tr:n_tr + n_va]
    test  += imgs[n_tr + n_va:]

# ---- 5) Spread background images across the splits too (~70/20/10) ----
for i, img in enumerate(negatives):
    (train if i % 10 < 7 else val if i % 10 < 9 else test).append(img)

# ---- 6) Copy image+label pairs into the YOLO folder layout ----
# Layout: dataset/<split>/images/*.jpg  and  dataset/<split>/labels/*.txt
if DATASET.exists():
    shutil.rmtree(DATASET)      # wipe so re-running gives a clean split

def write_split(imgs, split):
    img_dir = DATASET / split / 'images'
    lbl_dir = DATASET / split / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img in imgs:
        shutil.copy(img, img_dir / img.name)          # copy the image
        lf = all_labels.get(img.stem)
        if lf and lf.exists():
            shutil.copy(lf, lbl_dir / f'{img.stem}.txt')  # copy its label

for imgs, split in [(train, 'train'), (val, 'valid'), (test, 'test')]:
    write_split(imgs, split)

# ---- 7) Print class counts per split so you can sanity-check the balance ----
def split_class_counts(imgs):
    c = Counter()
    for img in imgs:
        if img in primary_class:
            c[primary_class[img]] += 1
    return {CLASSES[k]: c.get(k, 0) for k in range(len(CLASSES))}

for imgs, split in [(train, 'train'), (val, 'valid'), (test, 'test')]:
    print(f'{split:6s}: {len(imgs):3d} images  | classes: {split_class_counts(imgs)}')

# ---- 8) Write data.yaml (the file YOLO reads to find data + class names) ----
import yaml
data = {
    'path':  str(DATASET),       # dataset root
    'train': 'train/images',     # paths are relative to 'path'
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    len(CLASSES),        # number of classes (3)
    'names': CLASSES,             # carrot_peels, mixed_veg, onion
}
DATA_YAML.write_text(yaml.safe_dump(data, sort_keys=False))
print("\n✅ data.yaml:\n" + DATA_YAML.read_text())

In [ ]:
# ============================================================
# CELL 5 — Hyperparameter file (data augmentation + loss weights)
# ============================================================
# These control how aggressively training augments images and how the
# loss is weighted. Augmentation is kept moderate so vegetable textures
# aren't distorted beyond recognition. (Saved for reference; the actual
# training in CELL 8 passes its own augmentation values.)
hyp = {
  # --- optimizer schedule ---
  "lr0": 0.01,              # initial learning rate
  "lrf": 0.01,              # final LR fraction (for cosine schedule)
  "momentum": 0.937,
  "weight_decay": 0.0005,
  "warmup_epochs": 3.0,
  "warmup_momentum": 0.8,
  "warmup_bias_lr": 0.1,

  # --- data augmentation (kept gentle for food/vegetable images) ---
  "hsv_h": 0.015,           # hue jitter
  "hsv_s": 0.6,             # saturation jitter
  "hsv_v": 0.35,            # brightness jitter
  "degrees": 0.0,           # rotation
  "translate": 0.08,        # shift
  "scale": 0.3,             # zoom
  "shear": 2.0,
  "perspective": 0.0005,
  "flipud": 0.0,            # no vertical flip
  "fliplr": 0.5,            # 50% horizontal flip
  "mosaic": 0.7,            # mosaic augmentation (kept modest)
  "mixup": 0.05,            # mixup (kept small)

  # --- loss term weights ---
  "box": 7.5,               # bounding-box regression
  "cls": 0.5,               # classification
  "dfl": 1.5                # distribution focal loss
}

import yaml
HYP_YAML = CFG_DIR / "hyp_compost.yaml"
with open(HYP_YAML, "w") as f:
    yaml.safe_dump(hyp, f, sort_keys=False)

print(HYP_YAML.read_text())

In [ ]:
# ============================================================
# CELL 6 — Label integrity check
# ============================================================
# Before training, verify every label line is valid:
#   - exactly 5 values per line (class cx cy w h)
#   - the class ID is within range (0 .. number_of_classes - 1)
# This catches mislabeled files early, e.g. a class ID of 3 when you
# only have 3 classes (valid IDs are 0,1,2).
from glob import glob

bad_lines = 0
out_of_range = 0
max_cid = len(CLASSES) - 1   # highest valid class ID (= 2 for 3 classes)

# Real layout is dataset/<split>/labels/*.txt
for lf in glob(str(DATASET / "**" / "labels" / "*.txt"), recursive=True):
    with open(lf) as f:
        for line in f:
            p = line.strip().split()
            if len(p) != 5:
                print("Bad format line in:", lf)
                bad_lines += 1
                continue
            try:
                cid = int(float(p[0]))
                if cid < 0 or cid > max_cid:
                    print("Class ID out of range in:", lf)
                    out_of_range += 1
            except:
                bad_lines += 1

print("Bad-format lines:", bad_lines)
print("Out-of-range class IDs:", out_of_range)
# Stop here if any class ID is invalid — training would fail or mislabel.
assert out_of_range == 0, "Fix class IDs before training."

In [ ]:
# ============================================================
# CELL 7 — Verify the YAML files load correctly
# ============================================================
# A quick check that data.yaml and hyp_compost.yaml exist and parse,
# so we don't discover a typo only after training starts.
import yaml

print("DATA_YAML:", DATA_YAML, "exists:", DATA_YAML.exists(),
      "size:", DATA_YAML.stat().st_size if DATA_YAML.exists() else 0)
print("HYP_YAML:",  HYP_YAML,  "exists:", HYP_YAML.exists(),
      "size:", HYP_YAML.stat().st_size if HYP_YAML.exists() else 0)

# Show just the first 400 characters so the output isn't huge.
if DATA_YAML.exists():
    print("\n--- data.yaml head ---\n", DATA_YAML.read_text()[:400])
if HYP_YAML.exists():
    print("\n--- hyp.yaml head  ---\n", HYP_YAML.read_text()[:400])

# Confirm they parse as valid YAML.
if DATA_YAML.exists():
    _d = yaml.safe_load(DATA_YAML.read_text())
    print("\nParsed data.yaml OK. Keys:", list(_d.keys()))
if HYP_YAML.exists():
    _h = yaml.safe_load(HYP_YAML.read_text())
    print("Parsed hyp.yaml OK. Keys:", list(_h.keys()))

In [ ]:
# ============================================================
# CELL 8 — Train the model
# ============================================================
from ultralytics import YOLO, __version__ as yv

print("Ultralytics:", yv)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# --- basic settings ---
IMG_SZ   = 640                    # input image size
EPOCHS   = 150                    # training passes over the dataset
MODEL    = "yolo11n.pt"           # YOLOv8 nano (small/fast). NOTE: needs the "v".
DEVICE   = 0 if torch.cuda.is_available() else "cpu"   # GPU 0, else CPU
# Unique run name with a timestamp so runs don't overwrite each other.
RUN_NAME = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}_veg"

# --- load the pretrained model (transfer learning) ---
model = YOLO(MODEL)

# --- training arguments ---
overrides = dict(
    data=str(DATA_YAML),          # our dataset description
    imgsz=IMG_SZ,
    epochs=EPOCHS,
    batch=16,                     # images per step (lower if you hit OOM)
    workers=2,                    # data-loading threads
    device=DEVICE,
    project=str(RUNS),            # where results are saved
    name=RUN_NAME,
    pretrained=True,
    optimizer="AdamW",
    lr0=0.001,                    # initial learning rate
    cos_lr=True,                  # cosine LR schedule
    patience=30,                  # early stop if no improvement for 30 epochs
    amp=True,                     # mixed precision (faster on GPU)
    seed=SEED,
    cache=True,                   # cache images in RAM for speed

    # --- augmentations applied during training ---
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.5,
    fliplr=0.5,
    flipud=0.1,
    degrees=15.0,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

# --- run training, printing a full traceback if it crashes ---
try:
    train_res = model.train(**overrides)
    print("\n✅ Training done. Run directory:", RUNS / RUN_NAME)
except Exception:
    import traceback
    print("\n❌ Training crashed. Traceback:\n")
    traceback.print_exc()
    raise

In [ ]:
# ============================================================
# CELL 9 — Validate (on the validation split)
# ============================================================
# Measures accuracy (mAP, precision, recall) on the validation set,
# which the model saw during training only for early-stopping decisions.
val_res = model.val(
    data=str(DATA_YAML),
    imgsz=IMG_SZ,
    split="val",
    project=str(RUNS),
    name=f"{RUN_NAME}_val",
    device=0 if torch.cuda.is_available() else "cpu"
)
print(val_res)

In [ ]:
# ============================================================
# CELL 10 — Test (on the held-out test split)
# ============================================================
# Final, unbiased evaluation on images the model never trained on.
test_res = model.val(
    data=str(DATA_YAML),
    imgsz=IMG_SZ,
    split="test",
    project=str(RUNS),
    name=f"{RUN_NAME}_test",
    device=0 if torch.cuda.is_available() else "cpu"
)
print(test_res)

In [ ]:
# ============================================================
# CELL 11 — Inference on sample test images (up to 12)
# ============================================================
# Runs the trained model on a few test images and saves the annotated
# results (boxes + class names) into the runs folder so you can look.
from glob import glob

test_img_dir = DATASET / 'test' / 'images'

# Grab up to 12 sample images from the test set.
sample_imgs = sorted(
    Path(p) for ext in [".jpg", ".jpeg", ".png"]
    for p in glob(str(test_img_dir / f"**/*{ext}"), recursive=True)
)[:12]

print(f"Found {len(sample_imgs)} sample images")
print(f"Path: {test_img_dir}")
print(f"Exists: {test_img_dir.exists()}")
assert len(sample_imgs) > 0, f"No images found in {test_img_dir}"

pred = model.predict(
    source=[str(p) for p in sample_imgs],
    imgsz=IMG_SZ,
    conf=0.50,                    # min confidence to keep a detection
    iou=0.45,                     # NMS overlap threshold
    device=0 if torch.cuda.is_available() else "cpu",
    save=True,                    # save annotated images
    project=str(RUNS),
    name=f"{RUN_NAME}_pred_samples"
)
pred

In [ ]:
# ============================================================
# CELL 12 — Export to ONNX (for Raspberry Pi / CPU deployment)
# ============================================================
# Converts the best PyTorch weights (best.pt) into an .onnx file that
# runs with onnxruntime on devices without a GPU, like a Raspberry Pi.
from ultralytics import YOLO

BEST = RUNS / RUN_NAME / "weights" / "best.pt"
assert BEST.exists(), f"best.pt not found at {BEST}. Did training finish?"

print("✅ Found best.pt. Exporting to ONNX...")

export_args = dict(
    format="onnx",
    opset=13,          # widely compatible ONNX version
    dynamic=False,     # fixed input size = faster CPU inference
    simplify=True,     # simplify the graph (needs onnxsim)
    half=False         # FP32 (FP16 rarely helps on a Pi CPU)
)

export_path = YOLO(BEST).export(**export_args)

print("\n✅ Export complete!")
print("Output:", export_path)
print("\nExport details:")
print(f"- Format: {export_args['format']}")
print(f"- Opset:  {export_args['opset']}")
print(f"- Dynamic input shapes: {export_args['dynamic']}")
print(f"- Simplified graph: {export_args['simplify']}")
print(f"- Precision: {'FP16' if export_args['half'] else 'FP32'}")

print("\n💡 On the Raspberry Pi, run:")
print("   pip install onnxruntime opencv-python numpy")
print("   then load with onnxruntime.InferenceSession('best.onnx')")

In [ ]:
# ============================================================
# CELL 13 — Reusable inference helper (.pt and .onnx)
# ============================================================
# A small class so you can run predictions later with one line, whether
# you loaded the PyTorch (.pt) model on a PC/GPU or the .onnx model on a Pi.
from typing import Iterable, Union, List
from glob import glob
import cv2
import onnxruntime as ort

def _expand_sources(
    sources: Union[str, Path, Iterable[Union[str, Path]]],
    exts=(".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
) -> List[str]:
    """Turn a file, folder, or glob pattern into a sorted list of image paths."""
    if isinstance(sources, (str, Path)):
        sources = [sources]
    files: List[str] = []
    for s in sources:
        s = Path(s)
        if s.is_dir():                                  # folder -> search recursively
            for e in exts:
                files.extend(glob(str(s / f"**/*{e}"), recursive=True))
        elif any(str(s).lower().endswith(e) for e in exts):  # single image
            files.append(str(s))
        else:                                           # treat as glob pattern
            files.extend(glob(str(s), recursive=True))
    return sorted(set(files))

class InferenceRunner:
    """
    Unified YOLOv8 inference:
      - .pt  -> uses Ultralytics YOLO (PC / GPU)
      - .onnx-> uses onnxruntime (Raspberry Pi / CPU)
    Automatically uses CUDA if available.
    """
    def __init__(self, weights: Union[str, Path], device: Union[int, str, None] = None):
        self.weights = Path(weights)
        self.device = (0 if torch.cuda.is_available() else "cpu") if device is None else device

        if self.weights.suffix == ".onnx":
            print(f"🧠 Loading ONNX model: {self.weights}")
            providers = ["CUDAExecutionProvider"] if torch.cuda.is_available() else ["CPUExecutionProvider"]
            self.session = ort.InferenceSession(str(self.weights), providers=providers)
            self.model_type = "onnx"
        else:
            from ultralytics import YOLO
            print(f"🧠 Loading YOLOv8 model: {self.weights}")
            self.model = YOLO(str(self.weights))
            self.model_type = "yolo"

    def __call__(self, sources, imgsz=640, conf=0.25, iou=0.45,
                 save=True, project=RUNS, name="inference"):
        files = _expand_sources(sources)
        if not files:
            raise FileNotFoundError(f"No images found for sources={sources}")

        # --- PyTorch path (full pipeline incl. NMS + drawing) ---
        if self.model_type == "yolo":
            results = self.model.predict(
                source=files, imgsz=imgsz, conf=conf, iou=iou,
                device=self.device, save=save, project=str(project), name=name,
            )
            return results, Path(project) / name

        # --- ONNX path (simple preprocessing only; NMS/drawing not included) ---
        input_name = self.session.get_inputs()[0].name
        output_names = [o.name for o in self.session.get_outputs()]
        out_dir = Path(project) / name
        out_dir.mkdir(parents=True, exist_ok=True)

        results = []
        for img_path in files:
            img = cv2.imread(img_path)
            if img is None:
                continue
            img_resized = cv2.resize(img, (imgsz, imgsz))
            img_input = img_resized[:, :, ::-1].transpose(2, 0, 1) / 255.0  # BGR->RGB, HWC->CHW, scale
            img_input = np.expand_dims(img_input.astype(np.float32), 0)     # add batch dim
            preds = self.session.run(output_names, {input_name: img_input})
            results.append(preds)
            if save:
                cv2.imwrite(str(out_dir / Path(img_path).name), img_resized)
        return results, out_dir

In [ ]:
# ============================================================
# CELL 14 — Show the latest run's metrics + result plots
# ============================================================
from IPython.display import Image, display
import pandas as pd

# Pick the most recent run folder.
run_dir = sorted(RUNS.glob('run_*'))[-1]
print("Run folder:", run_dir)

# Print the final-epoch metrics from results.csv.
csv_path = run_dir / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    print("\n=== Final Metrics ===")
    print(df.tail(1)[['metrics/mAP50(B)',
                       'metrics/mAP50-95(B)',
                       'metrics/precision(B)',
                       'metrics/recall(B)']].to_string())

# Display the standard YOLO result plots if present.
for img_name in ['results.png',
                 'confusion_matrix_normalized.png',
                 'PR_curve.png',
                 'F1_curve.png']:
    img_path = run_dir / img_name
    if img_path.exists():
        print(f"\n=== {img_name} ===")
        display(Image(str(img_path)))

In [ ]:
# ============================================================
# CELL 15 — Compare metrics across all runs
# ============================================================
# Prints the final metrics of every run so you can see which settings
# worked best over time.
import pandas as pd

runs = sorted(RUNS.glob('run_*/results.csv'))
for run in runs:
    df = pd.read_csv(run)
    df.columns = df.columns.str.strip()
    final = df.tail(1)
    print(f"\n=== {run.parent.name} ===")
    print(f"mAP50:    {final['metrics/mAP50(B)'].values[0]:.4f}")
    print(f"mAP50-95: {final['metrics/mAP50-95(B)'].values[0]:.4f}")
    print(f"Precision:{final['metrics/precision(B)'].values[0]:.4f}")
    print(f"Recall:   {final['metrics/recall(B)'].values[0]:.4f}")

In [ ]:
# ============================================================
# CELL 16 (corrected) — save the TRAINING run (the one with weights)
# ============================================================
import shutil
from pathlib import Path

DRIVE_RUNS = Path('/content/drive/MyDrive/Machine Learning/compost_foodwaste/runs')
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

# pick the newest run that actually contains weights/best.pt
train_runs = sorted(RUNS.glob('*/weights/best.pt'), key=lambda p: p.stat().st_mtime)
assert train_runs, "No best.pt found — did training finish?"
latest_run = train_runs[-1].parent.parent          # .../run_..._veg

dest = DRIVE_RUNS / latest_run.name
shutil.copytree(str(latest_run), str(dest), dirs_exist_ok=True)

print(f"✅ Saved to Drive: {dest}")
print(f"   - best.pt: {(dest / 'weights' / 'best.pt').exists()}")
print(f"   - results.png: {(dest / 'results.png').exists()}")